In [1]:
# imports
import pandas as pd
from Bio.PDB import PDBParser
import os
import glob
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.metrics import jaccard_score
import lightgbm as lgb

In [2]:
# Amino acids constants (renaming from triple to single digits)

AA3_TO_AA1 = {
    "ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C",
    "GLN": "Q", "GLU": "E", "GLY": "G", "HIS": "H", "ILE": "I",
    "LEU": "L", "LYS": "K", "MET": "M", "PHE": "F", "PRO": "P",
    "SER": "S", "THR": "T", "TRP": "W", "TYR": "Y", "VAL": "V"
}

# feauture name constants

HYDROPHOBIC = {"A", "V", "I", "L", "M", "F", "W", "Y", "P"}
POLAR = {"S", "T", "N", "Q", "C", "G"}
POSITIVE = {"K", "R", "H"}
NEGATIVE = {"D", "E"}
AROMATIC = {"F", "W", "Y", "H"}

In [3]:
#convert label strings to sets

def parse_binding_string(s):
    """
    Wandelt einen String wie
    'A_24 A_25 A_26'
    in ein Set um:
    {'A_24', 'A_25', 'A_26'}
    """
    if pd.isna(s) or not str(s).strip():
        return set()
    return set(str(s).strip().split())


def load_train_labels(csv_path):
    """
    Lädt train.csv und erzeugt zusätzlich eine Spalte 'binding_set'.
    
    Erwartetes Format:
    id,residue
    0,A_24 A_25 A_26 ...
    1,A_14 A_15 A_16 ...
    """
    df = pd.read_csv(csv_path)

    # ID als String, damit '0' sauber zu Dateinamen gematcht werden kann
    df["id"] = df["id"].astype(str)

    # Pocket-Residuen in Sets umwandeln
    df["binding_set"] = df["resid"].apply(parse_binding_string)

    return df

In [4]:
# helper function for filenames

def extract_pdb_id(filepath):
    """
    Extrahiert aus einem Dateinamen wie:
    '0_protein.pdb'
    die ID:
    '0'
    """
    base = os.path.basename(filepath)           # z.B. '0_protein.pdb'
    name = os.path.splitext(base)[0]           # z.B. '0_protein'
    pdb_id = name.split("_")[0]                # z.B. '0'
    return pdb_id

In [5]:
# helper function for residue keys

def make_residue_key(chain_id, residue):
    """
    Baut einen Residue-Key im gleichen Format wie im CSV:
    - A_24
    - A_123_A
    
    residue.id ist typischerweise:
    (' ', 24, ' ')
    oder
    (' ', 123, 'A')
    """
    resid = residue.id[1]
    icode = residue.id[2].strip()

    if icode:
        return f"{chain_id}_{resid}_{icode}"
    return f"{chain_id}_{resid}"

In [6]:
# pasre single pdb file helper function
def parse_pdb_file(pdb_path, pdb_id=None):
    """
    Parst eine einzelne PDB-Datei und gibt eine Liste von Dictionaries zurück.
    
    Es werden nur Standard-Aminosäuren mit vorhandenem CA-Atom übernommen.
    """
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(pdb_id or "protein", pdb_path)

    rows = []

    # Nur erstes Modell verwenden
    model = next(structure.get_models())

    for chain in model:
        for residue in chain:
            # Nur Standard-Aminosäuren
            if residue.resname not in AA3_TO_AA1:
                continue

            # Nur Residuen mit Alpha-Carbon
            if "CA" not in residue:
                continue

            ca = residue["CA"].get_coord()

            row = {
                "pdb_id": str(pdb_id) if pdb_id is not None else None,
                "chain_id": chain.id,
                "residue_key": make_residue_key(chain.id, residue),
                "resname3": residue.resname,
                "aa": AA3_TO_AA1[residue.resname],
                "resid": residue.id[1],
                "icode": residue.id[2].strip(),
                "x": float(ca[0]),
                "y": float(ca[1]),
                "z": float(ca[2]),
            }

            rows.append(row)

    return rows

In [7]:
# parse all pdb files

def parse_all_pdbs(pdb_dir, max_files=None):
    """
    Parst alle .pdb-Dateien in einem Ordner.
    
    Erwartete Dateinamen:
    0_protein.pdb
    1_protein.pdb
    2_protein.pdb
    ...
    
    Gibt einen DataFrame mit allen Residuen zurück.
    """
    pdb_files = sorted(glob.glob(os.path.join(pdb_dir, "*.pdb")))
    all_rows = []

    if max_files is not None:
        pdb_files = pdb_files[:max_files]

    print(f"Gefundene PDB-Dateien: {len(pdb_files)}")

    for i, pdb_file in enumerate(pdb_files, start=1):
        pdb_id = extract_pdb_id(pdb_file)

        try:
            rows = parse_pdb_file(pdb_file, pdb_id=pdb_id)
            all_rows.extend(rows)
        except Exception as e:
            print(f"[FEHLER] {pdb_file}: {e}")

        if i % 100 == 0 or i == len(pdb_files):
            print(f"Verarbeitet: {i}/{len(pdb_files)}")

    df = pd.DataFrame(all_rows)
    return df

In [8]:
# validate parser for multiple files

def validate_parser_multiple(pdb_dir, labels_df, max_files=50):
    """
    Prüft für viele Dateien, ob alle Label-Residuen
    aus train.csv im jeweiligen PDB gefunden werden.
    """
    pdb_files = sorted(glob.glob(os.path.join(pdb_dir, "*.pdb")))

    if max_files is not None:
        pdb_files = pdb_files[:max_files]

    total_checked = 0
    files_with_missing = []

    for pdb_file in pdb_files:
        pdb_id = extract_pdb_id(pdb_file)

        # Nur validieren, wenn die ID auch in labels_df existiert
        if not (labels_df["id"] == pdb_id).any():
            print(f"[WARNUNG] Keine Label-Zeile für PDB-ID {pdb_id}")
            continue

        parsed_rows = parse_pdb_file(pdb_file, pdb_id=pdb_id)
        parsed_keys = {r["residue_key"] for r in parsed_rows}

        binding_set = labels_df.loc[labels_df["id"] == pdb_id, "binding_set"].iloc[0]
        missing = binding_set - parsed_keys

        total_checked += 1

        if len(missing) > 0:
            files_with_missing.append((pdb_id, sorted(missing)))
            print(f"[MISMATCH] {pdb_id}: fehlend {sorted(missing)}")

    print()
    print(f"Geprüfte Dateien: {total_checked}")
    print(f"Dateien mit fehlenden Label-Residuen: {len(files_with_missing)}")

    return files_with_missing


In [9]:
# parser inspection helper function 

def inspect_missing_residues(pdb_path, labels_df):
    pdb_id = extract_pdb_id(pdb_path)
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(pdb_id, pdb_path)
    model = next(structure.get_models())

    binding_set = labels_df.loc[labels_df["id"] == pdb_id, "binding_set"].iloc[0]

    all_residues = []
    parsed_keys = set()

    for chain in model:
        for residue in chain:
            hetfield, resseq, icode = residue.id
            icode = icode.strip()
            reskey = f"{chain.id}_{resseq}" if not icode else f"{chain.id}_{resseq}_{icode}"

            has_ca = "CA" in residue
            is_standard = residue.resname in AA3_TO_AA1

            all_residues.append({
                "chain_id": chain.id,
                "resname": residue.resname,
                "hetfield": hetfield,
                "resseq": resseq,
                "icode": icode,
                "residue_key": reskey,
                "has_ca": has_ca,
                "is_standard": is_standard,
            })

            if is_standard and has_ca:
                parsed_keys.add(reskey)

    all_df = pd.DataFrame(all_residues)
    missing = sorted(binding_set - parsed_keys)

    print(f"PDB-ID: {pdb_id}")
    print(f"Fehlende Label-Residuen: {missing}")
    print()

    for miss in missing:
        print("=" * 50)
        print("Fehlendes Label:", miss)

        parts = miss.split("_")
        chain_id = parts[0]
        resseq = int(parts[1])
        icode = parts[2] if len(parts) > 2 else ""

        candidates = all_df[
            (all_df["chain_id"] == chain_id) &
            (all_df["resseq"] == resseq)
        ].copy()

        if len(candidates) == 0:
            print("Kein Residuum mit gleicher chain/resseq im PDB gefunden.")
        else:
            print(candidates.sort_values(["chain_id", "resseq", "icode"]))
        print()
    

In [10]:
# another bigger helper function

def inspect_missing_residues_verbose(pdb_path, labels_df, window=10):
    pdb_id = extract_pdb_id(pdb_path)
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(pdb_id, pdb_path)
    model = next(structure.get_models())

    binding_set = labels_df.loc[labels_df["id"] == pdb_id, "binding_set"].iloc[0]

    all_residues = []

    for chain in model:
        for residue in chain:
            hetfield, resseq, icode = residue.id
            icode = icode.strip()
            reskey = f"{chain.id}_{resseq}" if not icode else f"{chain.id}_{resseq}_{icode}"

            all_residues.append({
                "chain_id": chain.id,
                "resname": residue.resname,
                "hetfield": hetfield,
                "resseq": resseq,
                "icode": icode,
                "residue_key": reskey,
                "has_ca": "CA" in residue,
                "is_standard": residue.resname in AA3_TO_AA1
            })

    all_df = pd.DataFrame(all_residues)

    parsed_keys = set(
        all_df.loc[
            all_df["is_standard"] & all_df["has_ca"],
            "residue_key"
        ]
    )

    missing = sorted(binding_set - parsed_keys)

    print(f"PDB-ID: {pdb_id}")
    print(f"Fehlende Label-Residuen: {missing}")
    print()

    print("Vorhandene Chains im Modell:")
    print(sorted(all_df["chain_id"].unique().tolist()))
    print()

    for miss in missing:
        print("=" * 60)
        print("Fehlendes Label:", miss)

        parts = miss.split("_")
        chain_id = parts[0]
        resseq = int(parts[1])
        icode = parts[2] if len(parts) > 2 else ""

        # Exakte Treffer
        exact = all_df[
            (all_df["chain_id"] == chain_id) &
            (all_df["resseq"] == resseq)
        ].copy()

        if len(exact) > 0:
            print("\nExakte Treffer in gleicher Chain/Resid:")
            print(exact.sort_values(["chain_id", "resseq", "icode"]))
        else:
            print("\nKein exaktes Residuum mit gleicher chain/resseq gefunden.")

        # Nachbar-Residuen in derselben Chain
        same_chain_near = all_df[
            (all_df["chain_id"] == chain_id) &
            (all_df["resseq"] >= resseq - window) &
            (all_df["resseq"] <= resseq + window)
        ].copy()

        print(f"\nResiduen in Chain {chain_id} im Bereich {resseq-window} bis {resseq+window}:")
        if len(same_chain_near) > 0:
            print(same_chain_near.sort_values(["resseq", "icode"])[
                ["chain_id", "resname", "resseq", "icode", "hetfield", "has_ca", "is_standard", "residue_key"]
            ])
        else:
            print("Keine Residuen in diesem Bereich gefunden.")

        # Gleiche Residuenummer in anderen Chains
        same_resseq_other_chains = all_df[
            (all_df["chain_id"] != chain_id) &
            (all_df["resseq"] == resseq)
        ].copy()

        print(f"\nGleiche Residuenummer {resseq} in anderen Chains:")
        if len(same_resseq_other_chains) > 0:
            print(same_resseq_other_chains.sort_values(["chain_id", "icode"])[
                ["chain_id", "resname", "resseq", "icode", "hetfield", "has_ca", "is_standard", "residue_key"]
            ])
        else:
            print("Keine Treffer mit gleicher Nummer in anderen Chains.")

        print()

In [11]:
# build train dataframe (OG)

def build_train_dataframe_OG(pdb_dir, labels_df, max_files=None):
    """
    Baut einen Trainings-DataFrame auf Residuum-Ebene.
    
    Jede Zeile = ein Residuum
    target = 1, wenn Residuum im Binding Pocket liegt
    target = 0 sonst
    """
    pdb_files = sorted(glob.glob(os.path.join(pdb_dir, "*.pdb")))

    if max_files is not None:
        pdb_files = pdb_files[:max_files]

    binding_map = dict(zip(labels_df["id"], labels_df["binding_set"]))

    all_rows = []

    for i, pdb_file in enumerate(pdb_files, start=1):
        pdb_id = extract_pdb_id(pdb_file)

        try:
            rows = parse_pdb_file(pdb_file, pdb_id=pdb_id)
        except Exception as e:
            print(f"[FEHLER] {pdb_file}: {e}")
            continue

        binding_set = binding_map.get(pdb_id, set())

        for row in rows:
            row["target"] = int(row["residue_key"] in binding_set)
            all_rows.append(row)

        if i % 100 == 0 or i == len(pdb_files):
            print(f"Train-Dateien verarbeitet: {i}/{len(pdb_files)}")

    train_df = pd.DataFrame(all_rows)
    return train_df

In [12]:
# build train dataframe verbesserte version

def build_train_dataframe(pdb_dir, labels_df, max_files=None, drop_bad_proteins=False,
                          max_missing_fraction=0.30, max_missing_abs=5):
    """
    Baut einen Trainings-DataFrame auf Residuum-Ebene.

    Jede Zeile = ein Residuum
    target = 1, wenn Residuum im Binding Pocket liegt
    target = 0 sonst

    Zusätzlich:
    - berücksichtigt nur Label-Residuen, die im PDB tatsächlich vorhanden sind
    - erstellt einen Report über fehlende Label-Residuen
    - kann problematische Proteine optional ausschließen
    """
    pdb_files = sorted(glob.glob(os.path.join(pdb_dir, "*.pdb")))

    if max_files is not None:
        pdb_files = pdb_files[:max_files]

    binding_map = dict(zip(labels_df["id"], labels_df["binding_set"]))

    all_rows = []
    missing_reports = []

    kept_proteins = 0
    dropped_proteins = 0

    for i, pdb_file in enumerate(pdb_files, start=1):
        pdb_id = extract_pdb_id(pdb_file)

        try:
            rows = parse_pdb_file(pdb_file, pdb_id=pdb_id)
        except Exception as e:
            print(f"[FEHLER] {pdb_file}: {e}")
            continue

        binding_set = binding_map.get(pdb_id, set())
        parsed_keys = {row["residue_key"] for row in rows}

        # nur Labels behalten, die in der Struktur wirklich existieren
        effective_binding_set = binding_set & parsed_keys
        missing_labels = sorted(binding_set - parsed_keys)

        n_labels = len(binding_set)
        n_missing = len(missing_labels)
        missing_fraction = (n_missing / n_labels) if n_labels > 0 else 0.0

        # Heuristik: Protein ist problematisch, wenn zu viele Label-Residuen fehlen
        is_problematic = (
            (n_missing > max_missing_abs) or
            (missing_fraction > max_missing_fraction)
        )

        missing_reports.append({
            "pdb_id": pdb_id,
            "n_labels": n_labels,
            "n_parsed_residues": len(parsed_keys),
            "n_effective_labels": len(effective_binding_set),
            "n_missing_labels": n_missing,
            "missing_fraction": missing_fraction,
            "is_problematic": int(is_problematic),
            "missing_labels": " ".join(missing_labels)
        })

        # optional: problematische Proteine komplett überspringen
        if drop_bad_proteins and is_problematic:
            dropped_proteins += 1
            if i % 100 == 0 or i == len(pdb_files):
                print(f"Train-Dateien verarbeitet: {i}/{len(pdb_files)}")
            continue

        for row in rows:
            row["target"] = int(row["residue_key"] in effective_binding_set)
            row["n_labels_total"] = n_labels
            row["n_missing_labels"] = n_missing
            row["missing_fraction"] = missing_fraction
            row["is_problematic_protein"] = int(is_problematic)
            all_rows.append(row)

        kept_proteins += 1

        if i % 100 == 0 or i == len(pdb_files):
            print(f"Train-Dateien verarbeitet: {i}/{len(pdb_files)}")

    train_df = pd.DataFrame(all_rows)
    missing_df = pd.DataFrame(missing_reports)

    print()
    print("Build-Train-DataFrame Zusammenfassung:")
    print(f"Behaltene Proteine: {kept_proteins}")
    print(f"Ausgeschlossene Proteine: {dropped_proteins}")
    print(f"Proteine gesamt im Report: {len(missing_df)}")

    return train_df, missing_df

In [13]:
# data frame analysis helper function

def analyze_missing_label_stats(missing_df):
    """
    Zeigt eine Übersicht über fehlende Label-Residuen pro Protein.
    """
    print()
    print("=" * 60)
    print("MISSING LABEL STATISTIK")
    print("=" * 60)

    print("\nDeskriptive Statistik:")
    print(missing_df[["n_labels", "n_missing_labels", "missing_fraction"]].describe())

    print("\nAnzahl Proteine mit mindestens 1 fehlendem Label:")
    print((missing_df["n_missing_labels"] > 0).sum())

    print("\nAnzahl problematischer Proteine:")
    print((missing_df["is_problematic"] == 1).sum())

    print("\nTop 20 problematische Proteine nach missing_fraction:")
    print(
        missing_df.sort_values(
            ["missing_fraction", "n_missing_labels"],
            ascending=False
        )[[
            "pdb_id",
            "n_labels",
            "n_missing_labels",
            "missing_fraction",
            "missing_labels"
        ]].head(20)
    )

In [14]:
# build test dataframe

def build_test_dataframe(pdb_dir, max_files=None):
    """
    Baut einen Test-DataFrame auf Residuum-Ebene.
    Noch ohne target.
    """
    test_df = parse_all_pdbs(pdb_dir, max_files=max_files)
    return test_df

In [15]:
# feature engineering (gathering info which i think is relevant)

def add_basic_aa_features(df):
    """
    Fügt einfache chemische Eigenschaften der Aminosäuren hinzu.
    """
    df = df.copy()

    df["is_hydrophobic"] = df["aa"].isin(HYDROPHOBIC).astype(int)
    df["is_polar"] = df["aa"].isin(POLAR).astype(int)
    df["is_positive"] = df["aa"].isin(POSITIVE).astype(int)
    df["is_negative"] = df["aa"].isin(NEGATIVE).astype(int)
    df["is_aromatic"] = df["aa"].isin(AROMATIC).astype(int)

    return df


def safe_mean_topk(sorted_distances, k):
    """
    Mittlere Distanz der k nächsten Nachbarn.
    Falls weniger als k Nachbarn da sind, wird automatisch angepasst.
    """
    k = min(k, sorted_distances.shape[1])
    if k == 0:
        return np.zeros(sorted_distances.shape[0], dtype=float)
    return sorted_distances[:, :k].mean(axis=1)


def add_geometric_features(df):
    """
    Berechnet pro Protein geometrische Features auf Basis der CA-Koordinaten.
    """
    all_groups = []

    for pdb_id, g in df.groupby("pdb_id", sort=False):
        g = g.copy().reset_index(drop=True)

        coords = g[["x", "y", "z"]].values

        # Proteinzentrum
        center = coords.mean(axis=0)
        g["dist_to_center"] = np.linalg.norm(coords - center, axis=1)

        # Paarweise Distanzmatrix
        dmat = np.linalg.norm(coords[:, None, :] - coords[None, :, :], axis=-1)

        # Eigenabstand ignorieren
        np.fill_diagonal(dmat, np.inf)

        # Anzahl Nachbarn innerhalb verschiedener Radien
        g["n_neighbors_6"] = (dmat < 6.0).sum(axis=1)
        g["n_neighbors_8"] = (dmat < 8.0).sum(axis=1)
        g["n_neighbors_10"] = (dmat < 10.0).sum(axis=1)

        # Distanzen zu den nächsten Nachbarn
        sorted_d = np.sort(dmat, axis=1)
        g["mean_knn_3"] = safe_mean_topk(sorted_d, 3)
        g["mean_knn_5"] = safe_mean_topk(sorted_d, 5)
        g["mean_knn_10"] = safe_mean_topk(sorted_d, 10)

        all_groups.append(g)

    return pd.concat(all_groups, ignore_index=True)


def quick_feature_check(df):
    """
    Zeigt einfache Mittelwerte der wichtigsten Features für target=0 und target=1.
    """
    cols = [
        "dist_to_center",
        "n_neighbors_6",
        "n_neighbors_8",
        "n_neighbors_10",
        "mean_knn_3",
        "mean_knn_5",
        "mean_knn_10",
    ]

    print("\nMittelwerte nach target:\n")
    print(df.groupby("target")[cols].mean())

    print("\nAnzahl positive Residuen:", int(df["target"].sum()))
    print("Anzahl negative Residuen:", int((df["target"] == 0).sum()))

In [16]:
# model training

def train_lgbm(train_df, feature_cols, n_splits=5):
    X = train_df[feature_cols]
    y = train_df["target"]
    groups = train_df["pdb_id"]

    gkf = GroupKFold(n_splits=n_splits)

    oof_preds = np.zeros(len(train_df))
    models = []

    for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
        print(f"\n=== Fold {fold} ===")

        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = lgb.LGBMClassifier(
            n_estimators=500,
            learning_rate=0.05,
            num_leaves=63,
            subsample=0.8,
            colsample_bytree=0.8,
            class_weight="balanced",
            random_state=42
        )

        model.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            eval_metric="binary_logloss",
            categorical_feature=["aa", "chain_id"]
        )

        preds = model.predict_proba(X_val)[:, 1]
        oof_preds[val_idx] = preds
        models.append(model)

    train_df = train_df.copy()
    train_df["pred"] = oof_preds

    return models, train_df

In [17]:
# IOU / treshold calculation

def compute_iou(pred_set, true_set):
    union = pred_set | true_set
    if len(union) == 0:
        return 1.0
    return len(pred_set & true_set) / len(union)


def evaluate_threshold(df, threshold):
    ious = []

    for pdb_id, g in df.groupby("pdb_id"):
        pred_res = set(g.loc[g["pred"] >= threshold, "residue_key"])
        true_res = set(g.loc[g["target"] == 1, "residue_key"])

        if len(pred_res) == 0:
            top = g.sort_values("pred", ascending=False).iloc[0]["residue_key"]
            pred_res = {top}

        iou = compute_iou(pred_res, true_res)
        ious.append(iou)

    return np.mean(ious)


def find_best_threshold(df):
    best_thr = 0.5
    best_score = 0.0

    for t in np.linspace(0.05, 0.95, 30):
        score = evaluate_threshold(df, t)
        print(f"t={t:.2f} -> IoU={score:.4f}")

        if score > best_score:
            best_score = score
            best_thr = t

    print("\nBESTER THRESHOLD:")
    print(f"threshold={best_thr:.2f}, IoU={best_score:.4f}")

    return best_thr, best_score

In [18]:
# official metric / submission helpers

def evaluate_official_metric(df, threshold):
    """
    Bewertet einen Threshold mit der offiziellen Metrik-Logik
    auf Residuum-Ebene (globaler Jaccard über alle Residuen).
    """
    tmp = df.copy()
    tmp["prediction"] = (tmp["pred"] >= threshold).astype(int)

    y_true = tmp["target"].astype(int).values
    y_pred = tmp["prediction"].astype(int).values

    return jaccard_score(y_true, y_pred)


def find_best_threshold_official(df):
    """
    Sucht den besten Threshold anhand der offiziellen Metrik.
    """
    best_thr = 0.5
    best_score = -1.0

    for t in np.linspace(0.05, 0.95, 30):
        score = evaluate_official_metric(df, t)
        print(f"t={t:.2f} -> Official Jaccard={score:.4f}")

        if score > best_score:
            best_score = score
            best_thr = t

    print("\nBESTER THRESHOLD (OFFICIAL METRIC):")
    print(f"threshold={best_thr:.2f}, Jaccard={best_score:.4f}")

    return best_thr, best_score

In [19]:
# build submission

def build_submission(test_df, threshold, output_path="submission.csv"):
    """
    Baut eine Submission im Format:
    id,prediction

    prediction = leerzeichen-getrennte residue_key-Werte
    """
    df = test_df.copy()
    df["is_pred"] = (df["pred"] >= threshold).astype(int)

    submission_rows = []

    for pdb_id, g in df.groupby("pdb_id"):
        predicted_residues = g.loc[g["is_pred"] == 1, "residue_key"].tolist()

        # Fallback: mindestens 1 Residuum vorhersagen
        if len(predicted_residues) == 0:
            top_res = g.sort_values("pred", ascending=False).iloc[0]["residue_key"]
            predicted_residues = [top_res]

        prediction_str = " ".join(predicted_residues)

        submission_rows.append({
            "id": pdb_id,
            "prediction": prediction_str
        })

    submission_df = pd.DataFrame(submission_rows)
    submission_df = submission_df.sort_values("id").reset_index(drop=True)
    submission_df.to_csv(output_path, index=False)

    print(f"Submission gespeichert unter: {output_path}")
    print(submission_df.head())

    return submission_df

In [20]:
# predictions

def predict_test(models, test_df, feature_cols):
    X_test = test_df[feature_cols].copy()

    preds = np.zeros(len(test_df), dtype=float)

    for model in models:
        preds += model.predict_proba(X_test)[:, 1]

    preds /= len(models)

    test_df = test_df.copy()
    test_df["pred"] = preds

    return test_df

In [21]:
if __name__ == "__main__":
    # -------------------------
    # Pfade anpassen
    # -------------------------

    # Pfade Laptop
    #TRAIN_CSV_PATH = "C:/Users/Lukas/Master_AI/2nd_Semester/KV_Structural_Bioinformatics/train.csv"
    #TRAIN_PDB_DIR = "C:/Users/Lukas/Master_AI/2nd_Semester/KV_Structural_Bioinformatics/train"
    #TEST_PDB_DIR = "C:/Users/Lukas/Master_AI/2nd_Semester/KV_Structural_Bioinformatics/test"

    # Pfade PC
    TRAIN_CSV_PATH = "C:/Users/stec/PycharmProjects/Master_AI/2nd_Semester/KV_Structural_Bioinformatics/train.csv"
    TRAIN_PDB_DIR = "C:/Users/stec/PycharmProjects/Master_AI/2nd_Semester/KV_Structural_Bioinformatics/train"
    TEST_PDB_DIR = "C:/Users/stec/PycharmProjects/Master_AI/2nd_Semester/KV_Structural_Bioinformatics/test"

    # -------------------------
    # Labels laden
    # -------------------------
    print("Lade train.csv ...")
    train_labels = load_train_labels(TRAIN_CSV_PATH)

    print()
    print("Erste Zeilen aus train.csv:")
    print(train_labels.head())

    print()
    print("Beispiel binding_set für ID 0:")
    print(train_labels.loc[train_labels["id"] == "0", "binding_set"].iloc[0])

    # -------------------------
    # Parser über mehrere Dateien prüfen
    # -------------------------
    print()
    print("=" * 60)
    print("VALIDIERUNG ÜBER MEHRERE DATEIEN")
    print("=" * 60)
    problems = validate_parser_multiple(TRAIN_PDB_DIR, train_labels, max_files=None)

    # ----------
    # Inspection of some proteins
    # -----------

    inspect_missing_residues(os.path.join(TRAIN_PDB_DIR, "10043_protein.pdb"), train_labels)
    inspect_missing_residues(os.path.join(TRAIN_PDB_DIR, "10182_protein.pdb"), train_labels)
    inspect_missing_residues(os.path.join(TRAIN_PDB_DIR, "10307_protein.pdb"), train_labels)
    inspect_missing_residues(os.path.join(TRAIN_PDB_DIR, "10381_protein.pdb"), train_labels)

    print()

    inspect_missing_residues_verbose(os.path.join(TRAIN_PDB_DIR, "10043_protein.pdb"), train_labels)
    inspect_missing_residues_verbose(os.path.join(TRAIN_PDB_DIR, "10182_protein.pdb"), train_labels)
    inspect_missing_residues_verbose(os.path.join(TRAIN_PDB_DIR, "10307_protein.pdb"), train_labels)
    inspect_missing_residues_verbose(os.path.join(TRAIN_PDB_DIR, "10381_protein.pdb"), train_labels)


    # -------------------------
    # Trainings-DataFrame bauen
    # -------------------------
    print()
    print("=" * 60)
    print("TRAIN-DATAFRAME BAUEN")
    print("=" * 60)
    train_df, missing_df = build_train_dataframe(
        TRAIN_PDB_DIR,
        train_labels,
        max_files=None,
        drop_bad_proteins=False,
        max_missing_fraction=0.30,
        max_missing_abs=5
    )

    print()
    print("Anzahl Dateien mit fehlenden Label-Residuen:")
    print(len(missing_df))

    missing_df.to_csv("missing_labels_report.csv", index=False)

    print()
    print("Train-DataFrame Shape:")
    print(train_df.shape)

    print()
    print("Erste Zeilen:")
    print(train_df.head())

    print()
    print("Verteilung target:")
    print(train_df["target"].value_counts())

    print()
    print("Anzahl positive Residuen:")
    print(train_df["target"].sum())

    # Missing-Report analysieren
    analyze_missing_label_stats(missing_df)

    # -------------------------
    # RAM OPTIMIZATION (WICHTIG)
    # -------------------------
    train_df = train_df.drop(columns=[
        "n_labels_total",
        "n_missing_labels",
        "missing_fraction",
        "is_problematic_protein"
    ])

    print()
    print("Train-DataFrame nach Cleanup:")
    print(train_df.shape)

    # -------------------------
    # Features berechnen
    # -------------------------
    print()
    print("=" * 60)
    print("FEATURES BERECHNEN")
    print("=" * 60)

    train_df = add_basic_aa_features(train_df)
    train_df = add_geometric_features(train_df)

    print()
    print("Train-DataFrame mit Features:")
    print(train_df.head())

    quick_feature_check(train_df)

    # -------------------------
    # Feature-Spalten fürs Modell
    # -------------------------
    feature_cols = [
        "aa",
        "chain_id",
        "is_hydrophobic",
        "is_polar",
        "is_positive",
        "is_negative",
        "is_aromatic",
        "dist_to_center",
        "n_neighbors_6",
        "n_neighbors_8",
        "n_neighbors_10",
        "mean_knn_3",
        "mean_knn_5",
        "mean_knn_10",
    ]

    # Kategorien für LightGBM setzen
    train_df["aa"] = train_df["aa"].astype("category")
    train_df["chain_id"] = train_df["chain_id"].astype("category")

    # -------------------------
    # Modell trainieren
    # -------------------------
    print()
    print("=" * 60)
    print("MODEL TRAINING")
    print("=" * 60)

    models, train_df = train_lgbm(train_df, feature_cols, n_splits=5)

    print()
    print("OOF Predictions hinzugefügt:")
    print(train_df[["pdb_id", "residue_key", "target", "pred"]].head())

    # -------------------------
    # Threshold für IoU optimieren (kann gelöscht werden i guess)
    # -------------------------
    print()
    print("=" * 60)
    print("THRESHOLD OPTIMIZATION")
    print("=" * 60)

    best_thr, best_iou = find_best_threshold(train_df)

    print()
    print("FERTIG")
    print(f"Bester Threshold: {best_thr:.2f}")
    print(f"Beste mittlere IoU: {best_iou:.4f}")


    best_thr_iou, best_iou = find_best_threshold(train_df)

    print()
    print("Bester Threshold nach proteinweiser Mean-IoU:")
    print(f"Threshold: {best_thr_iou:.2f}")
    print(f"Mean IoU: {best_iou:.4f}")

    # -------------------------
    # Threshold für offizielle Metrik optimieren (OFFICIAL)
    # -------------------------
    print()
    print("=" * 60)
    print("THRESHOLD OPTIMIZATION (OFFICIAL METRIC)")
    print("=" * 60)

    best_thr_official, best_score_official = find_best_threshold_official(train_df)

    print()
    print("Bester Threshold nach offizieller Metrik:")
    print(f"Threshold: {best_thr_official:.2f}")
    print(f"Official Jaccard: {best_score_official:.4f}")

    # -------------------------
    # TEST-DATAFRAME BAUEN
    # -------------------------
    print()
    print("=" * 60)
    print("TEST-DATAFRAME BAUEN")
    print("=" * 60)

    test_df = build_test_dataframe(TEST_PDB_DIR, max_files= None)

    print("Test-DataFrame Shape:")
    print(test_df.shape)

    # -------------------------
    # TEST FEATURES
    # -------------------------
    print()
    print("=" * 60)
    print("TEST FEATURES")
    print("=" * 60)

    test_df = add_basic_aa_features(test_df)
    test_df = add_geometric_features(test_df)

    test_df["aa"] = test_df["aa"].astype("category")
    test_df["chain_id"] = test_df["chain_id"].astype("category")

    # -------------------------
    # TEST PREDICTION
    # -------------------------
    print()
    print("=" * 60)
    print("TEST PREDICTION")
    print("=" * 60)

    test_df = predict_test(models, test_df, feature_cols)

    print(test_df[["pdb_id", "residue_key", "pred"]].head())


    # -------------------------
    # SUBMISSION ERZEUGEN
    # -------------------------
    print()
    print("=" * 60)
    print("SUBMISSION ERZEUGEN")
    print("=" * 60)

    submission_df = build_submission(
        test_df,
        threshold=best_thr_official,
        output_path="submission.csv"
    )

    print()
    print("FERTIG")
    print(f"Verwendeter offizieller Threshold: {best_thr_official:.2f}")
    print("Submission-Datei: submission.csv")

    print(submission_df.shape)
    print(submission_df.head())
    print(submission_df.columns)

Lade train.csv ...

Erste Zeilen aus train.csv:
  id                                              resid  \
0  0  A_24 A_25 A_26 A_27 A_28 A_29 A_30 A_31 A_32 A...   
1  1  A_14 A_15 A_16 A_28 A_30 A_31 A_54 A_87 A_89 A...   
2  2  A_249 A_255 A_259 A_268 A_270 A_277 A_278 A_27...   
3  3  C_142 C_143 C_144 C_146 C_147 C_148 C_507 C_50...   
4  4  A_162 A_163 A_164 A_201 A_204 A_206 A_207 A_22...   

                                         binding_set  
0  {A_108, A_398, A_348, A_324, A_32, A_250, A_35...  
1  {A_108, A_164, A_140, A_141, A_31, A_162, A_14...  
2  {A_323, A_293, A_288, A_348, A_365, A_339, A_4...  
3  {C_565, C_761, F_514, C_688, C_760, C_143, C_6...  
4  {A_299, A_269, A_164, A_396, A_228, A_309, B_1...  

Beispiel binding_set für ID 0:
{'A_108', 'A_398', 'A_348', 'A_324', 'A_32', 'A_250', 'A_359', 'A_31', 'A_352', 'A_25', 'A_38', 'A_405', 'A_35', 'A_104', 'A_33', 'A_114', 'A_29', 'A_403', 'A_318', 'A_317', 'A_43', 'A_259', 'A_409', 'A_401', 'A_34', 'A_28', 'A_148', '